In [4]:
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

url = URL.create(
    drivername="postgresql",
    username="asabirov",
    password="Asabirov@Db5955",
    host="thomas.proxy.rlwy.net",
    port=51432,
    database="academy_db",
)
engine = create_engine(url)

with engine.connect() as conn:
    tables = pd.read_sql(text("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public' AND table_name LIKE 'seg_%'
        ORDER BY table_name
    """), conn)
    print(tables)

    print("\n" + "="*60)

    for t in tables['table_name']:
        cnt = pd.read_sql(text(f"SELECT COUNT(*) AS n FROM {t}"), conn)['n'][0]
        cols = pd.read_sql(text(f"""
            SELECT column_name, data_type
            FROM information_schema.columns
            WHERE table_schema = 'public' AND table_name = '{t}'
            ORDER BY ordinal_position
        """), conn)
        print(f"\n--- {t} ({cnt} строк) ---")
        print(cols.to_string(index=False))

                 table_name
0              seg_balances
1             seg_customers
2         seg_customers_360
3              seg_products
4       seg_revenue_monthly
5               seg_rfm_old
6        seg_tariff_current
7                  seg_temp
8          seg_transactions
9  seg_transactions_archive


--- seg_balances (427598 строк) ---
    column_name data_type
    customer_id    bigint
    month_start      date
avg_balance_uzs    bigint

--- seg_customers (20000 строк) ---
column_name data_type
customer_id    bigint
  full_name      text
 birth_date      date
     gender      text
     region      text
signup_date      date
tariff_code      text

--- seg_customers_360 (20000 строк) ---
   column_name data_type
   customer_id    bigint
        region      text
   signup_date      date
   crm_segment      text
 txn_cnt_total    bigint
turnover_total   numeric

--- seg_products (29983 строк) ---
 column_name data_type
 customer_id    bigint
product_code      text
   opened_at    

In [ ]:
import pandas as pd
from sqlalchemy import text

with engine.connect() as conn:
    # диапазоны дат
    for t, col in [("seg_transactions", "txn_ts"), ("seg_transactions_archive", "txn_ts"),
                   ("seg_balances", "month_start"), ("seg_revenue_monthly", "month_start")]:
        r = pd.read_sql(text(f"SELECT MIN({col}) AS min_d, MAX({col}) AS max_d, COUNT(DISTINCT customer_id) AS n_cust FROM {t}"), conn)
        print(f"{t}: {r.to_string(index=False)}")

    print()
    # пересечение txn_id между transactions и archive
    overlap = pd.read_sql(text("""
        SELECT COUNT(*) AS n_overlap
        FROM seg_transactions t
        JOIN seg_transactions_archive a ON t.txn_id = a.txn_id
    """), conn)
    print("Пересечение txn_id (transactions ∩ archive):", overlap['n_overlap'][0])

    # пересечение по customer_id
    overlap_cust = pd.read_sql(text("""
        SELECT COUNT(DISTINCT t.customer_id) AS n
        FROM seg_transactions t
        JOIN seg_transactions_archive a ON t.customer_id = a.customer_id
    """), conn)
    print("Клиентов есть в обеих:", overlap_cust['n'][0])

    # signup_date распределение
    signup = pd.read_sql(text("""
        SELECT date_trunc('month', signup_date) AS m, COUNT(*) AS n
        FROM seg_customers
        GROUP BY 1 ORDER BY 1
    """), conn)
    print("\nsignup_date по месяцам:")
    print(signup.to_string(index=False))

seg_transactions:               min_d               max_d  n_cust
2024-01-01 07:00:00 2025-12-28 23:59:00   19755
seg_transactions_archive:               min_d               max_d  n_cust
2024-01-01 07:00:00 2024-06-28 23:59:00   15962
seg_balances:      min_d      max_d  n_cust
2024-01-01 2025-12-01   20000
seg_revenue_monthly:      min_d      max_d  n_cust
2024-01-01 2025-12-01   20000

Пересечение txn_id (transactions ∩ archive): 2718367


In [3]:
import pandas as pd
from sqlalchemy import text

with engine.connect() as conn:
    signup = pd.read_sql(text("""
        SELECT date_trunc('month', signup_date) AS m, COUNT(*) AS n
        FROM seg_customers
        GROUP BY 1 ORDER BY 1
    """), conn)
    print(signup.to_string(index=False))

    # заодно: сколько клиентов вообще без единой транзакции
    no_txn = pd.read_sql(text("""
        SELECT COUNT(*) AS n
        FROM seg_customers c
        LEFT JOIN seg_transactions t ON c.customer_id = t.customer_id
        WHERE t.customer_id IS NULL
    """), conn)
    print("\nКлиентов без транзакций:", no_txn['n'][0])

    # и разброс "стажа" в месяцах на дату конца данных (2025-12-28)
    tenure = pd.read_sql(text("""
        SELECT
            (DATE '2025-12-28' - signup_date) / 30 AS tenure_months_approx
        FROM seg_customers
    """), conn)
    print("\nСтаж клиентов (мес, грубо):")
    print(tenure['tenure_months_approx'].describe())

                        m   n
2019-01-01 00:00:00+00:00 243
2019-02-01 00:00:00+00:00 228
2019-03-01 00:00:00+00:00 236
2019-04-01 00:00:00+00:00 256
2019-05-01 00:00:00+00:00 250
2019-06-01 00:00:00+00:00 228
2019-07-01 00:00:00+00:00 257
2019-08-01 00:00:00+00:00 244
2019-09-01 00:00:00+00:00 256
2019-10-01 00:00:00+00:00 224
2019-11-01 00:00:00+00:00 236
2019-12-01 00:00:00+00:00 237
2020-01-01 00:00:00+00:00 252
2020-02-01 00:00:00+00:00 245
2020-03-01 00:00:00+00:00 257
2020-04-01 00:00:00+00:00 198
2020-05-01 00:00:00+00:00 238
2020-06-01 00:00:00+00:00 267
2020-07-01 00:00:00+00:00 257
2020-08-01 00:00:00+00:00 211
2020-09-01 00:00:00+00:00 275
2020-10-01 00:00:00+00:00 243
2020-11-01 00:00:00+00:00 251
2020-12-01 00:00:00+00:00 245
2021-01-01 00:00:00+00:00 253
2021-02-01 00:00:00+00:00 244
2021-03-01 00:00:00+00:00 232
2021-04-01 00:00:00+00:00 243
2021-05-01 00:00:00+00:00 241
2021-06-01 00:00:00+00:00 239
2021-07-01 00:00:00+00:00 233
2021-08-01 00:00:00+00:00 244
2021-09-01

In [4]:
import pandas as pd
from sqlalchemy import text

with engine.connect() as conn:
    # сколько месяцев данных по балансам реально есть у каждого клиента
    months_per_cust = pd.read_sql(text("""
        SELECT customer_id, COUNT(*) AS n_months,
               MIN(month_start) AS first_month, MAX(month_start) AS last_month
        FROM seg_balances
        GROUP BY customer_id
    """), conn)
    print("Распределение количества месяцев в seg_balances на клиента:")
    print(months_per_cust['n_months'].describe())
    print()
    print("Уникальные значения n_months и их частота:")
    print(months_per_cust['n_months'].value_counts().sort_index())

    # связь между signup_date и первым месяцем в балансах (для тех, кто пришёл после 2024-01)
    check = pd.read_sql(text("""
        SELECT c.customer_id, c.signup_date, b.first_month
        FROM seg_customers c
        JOIN (SELECT customer_id, MIN(month_start) AS first_month FROM seg_balances GROUP BY customer_id) b
          ON c.customer_id = b.customer_id
        WHERE c.signup_date >= '2024-01-01'
        LIMIT 20
    """), conn)
    print("\nПример: клиенты с signup_date после 2024-01-01 vs первый месяц в balances:")
    print(check.to_string(index=False))

Распределение количества месяцев в seg_balances на клиента:
count    20000.000000
mean        21.379900
std          5.398633
min          3.000000
25%         24.000000
50%         24.000000
75%         24.000000
max         24.000000
Name: n_months, dtype: float64

Уникальные значения n_months и их частота:
n_months
3        13
4       269
5       247
6       242
7       224
8       263
9       247
10      288
11      223
12      245
13      227
14      237
15      248
16      246
17      272
18      217
19      261
20      249
21      250
22      265
23      254
24    15013
Name: count, dtype: int64

Пример: клиенты с signup_date после 2024-01-01 vs первый месяц в balances:
 customer_id signup_date first_month
           1  2024-03-19  2024-03-01
           4  2025-04-18  2025-04-01
           7  2024-01-24  2024-01-01
          10  2024-03-16  2024-03-01
          13  2025-07-03  2025-07-01
          14  2025-09-05  2025-09-01
          15  2025-05-13  2025-05-01
          24  2024

In [24]:
import pandas as pd
from sqlalchemy import text
import numpy as np

with engine.connect() as conn:
    # берём клиентов с полной историей (24 месяца) для чистоты анализа стабильности
    monthly_spend = pd.read_sql(text("""
        SELECT customer_id, date_trunc('month', txn_ts)::date AS month_start,
               SUM(CASE WHEN direction = 'debit' THEN amount_uzs ELSE 0 END) AS spend_uzs,
               COUNT(*) AS txn_cnt
        FROM seg_transactions
        GROUP BY customer_id, date_trunc('month', txn_ts)::date
    """), conn)

print("Строк:", len(monthly_spend))
print(monthly_spend.head())

# для клиентов с полными 24 месяцами считаем волатильность
full_hist_ids = months_per_cust = None  # заглушка, заполним ниже

pivot = monthly_spend.pivot(index='customer_id', columns='month_start', values='spend_uzs').fillna(0)
full_hist = pivot[pivot.gt(0).sum(axis=1) >= 20]  # почти все месяцы с активностью

cv = full_hist.std(axis=1) / (full_hist.mean(axis=1) + 1)
print("\nКоэффициент вариации помесячных трат (у клиентов с активной историей):")
print(cv.describe())

# доля клиентов с очень высокой волатильностью (CV > 1.5) - подозрение на смену поведения
print("\nДоля клиентов с CV > 1.5:", (cv > 1.5).mean())
print("Доля клиентов с CV > 1.0:", (cv > 1.0).mean())

KeyboardInterrupt: 

In [6]:
import pandas as pd
from sqlalchemy import text

with engine.connect() as conn:
    for col in ['direction', 'channel', 'mcc_group']:
        vals = pd.read_sql(text(f"""
            SELECT {col}, COUNT(*) AS n, AVG(amount_uzs) AS avg_amt
            FROM seg_transactions
            GROUP BY {col}
            ORDER BY n DESC
        """), conn)
        print(f"\n--- {col} ---")
        print(vals.to_string(index=False))

    amt = pd.read_sql(text("SELECT amount_uzs FROM seg_transactions LIMIT 100000"), conn)
    print("\namount_uzs describe:")
    print(amt['amount_uzs'].describe())
    print("Доля отрицательных:", (amt['amount_uzs'] < 0).mean())


--- direction ---
direction        n      avg_amt
      out 10841168 4.473848e+05
       in  1294964 2.747637e+06

--- channel ---
channel       n      avg_amt
    pos 4976269 4.633383e+05
   ecom 3458477 3.998999e+05
    atm 2406422 4.826390e+05
    p2p 1294964 2.747637e+06

--- mcc_group ---
    mcc_group       n      avg_amt
         cash 2406422 4.826390e+05
      grocery 1477469 4.196823e+05
     transfer 1294964 2.747637e+06
  marketplace 1211765 4.359793e+05
    transport 1099544 3.282771e+05
  restaurants  979872 2.023038e+05
     services  887458 5.590498e+05
         fuel  556254 6.491439e+05
entertainment  488703 2.017372e+05
  electronics  415623 8.600762e+05
    utilities  367839 3.741632e+05
     clothing  265194 5.748738e+05
     pharmacy  255117 3.646126e+05
        other  175814 3.500067e+05
    education  162091 7.286648e+05
       travel   92003 1.142929e+06

amount_uzs describe:
count    1.000000e+05
mean     7.321649e+05
std      2.752414e+06
min      1.000000e+04

In [ ]:
import pandas as pd
from sqlalchemy import text
import numpy as np

with engine.connect() as conn:
    monthly_spend = pd.read_sql(text("""
        SELECT customer_id, date_trunc('month', txn_ts)::date AS month_start,
               SUM(CASE WHEN direction = 'out' THEN amount_uzs ELSE 0 END) AS spend_uzs,
               COUNT(*) AS txn_cnt
        FROM seg_transactions
        GROUP BY customer_id, date_trunc('month', txn_ts)::date
    """), conn)

pivot = monthly_spend.pivot(index='customer_id', columns='month_start', values='spend_uzs').fillna(0)

# клиенты с длинной активной историей (>=20 месяцев с ненулевыми тратами)
full_hist = pivot[pivot.gt(0).sum(axis=1) >= 20]
print("Клиентов с активной историей >=20 мес:", len(full_hist))

cv = full_hist.std(axis=1) / (full_hist.mean(axis=1) + 1)
print("\nКоэффициент вариации помесячных трат:")
print(cv.describe())
print("\nДоля клиентов с CV > 1.5:", (cv > 1.5).mean())
print("Доля клиентов с CV > 1.0:", (cv > 1.0).mean())

# сравнение первой и второй половины истории (детект смены поведения)
first_half = full_hist.iloc[:, :12].mean(axis=1)
second_half = full_hist.iloc[:, 12:].mean(axis=1)
ratio = (second_half + 1) / (first_half + 1)
print("\nОтношение второй половины к первой (траты):")
print(ratio.describe())
print("Доля клиентов, у кого траты выросли >2x или упали <0.5x:", ((ratio > 2) | (ratio < 0.5)).mean())

# выбросы по общему обороту
total_turnover = pivot.sum(axis=1).sort_values(ascending=False)
print("\nТоп-10 клиентов по обороту:")
print(total_turnover.head(10))
print("\n99-й, 99.9-й перцентили оборота:")
print(total_turnover.quantile([0.9, 0.99, 0.999]))
print("Медиана оборота:", total_turnover.median())

In [8]:
print("Доля клиентов с CV > 1.5:", (cv > 1.5).mean())
print("Доля клиентов с CV > 1.0:", (cv > 1.0).mean())

first_half = full_hist.iloc[:, :12].mean(axis=1)
second_half = full_hist.iloc[:, 12:].mean(axis=1)
ratio = (second_half + 1) / (first_half + 1)
print("\nОтношение второй половины к первой (траты):")
print(ratio.describe())
print("Доля клиентов, у кого траты выросли >2x или упали <0.5x:", ((ratio > 2) | (ratio < 0.5)).mean())

total_turnover = pivot.sum(axis=1).sort_values(ascending=False)
print("\nТоп-10 клиентов по обороту:")
print(total_turnover.head(10))
print("\n99-й, 99.9-й перцентили оборота:")
print(total_turnover.quantile([0.9, 0.99, 0.999]))
print("Медиана оборота:", total_turnover.median())

Доля клиентов с CV > 1.5: 0.00304898256777358
Доля клиентов с CV > 1.0: 0.0312852124345463

Отношение второй половины к первой (траты):
count    15087.000000
mean         1.524914
std         13.395761
min          0.001345
25%          0.884101
50%          1.010424
75%          1.164095
max       1215.493809
dtype: float64
Доля клиентов, у кого траты выросли >2x или упали <0.5x: 0.059190031152647975

Топ-10 клиентов по обороту:
customer_id
9289     1.724049e+11
13706    3.169182e+10
2641     3.045703e+10
8782     2.566664e+10
19721    2.488381e+10
18629    2.410460e+10
6969     1.848038e+10
10584    1.767040e+10
11376    1.567691e+10
17748    1.466755e+10
dtype: float64

99-й, 99.9-й перцентили оборота:
0.900    4.777022e+08
0.990    2.139199e+09
0.999    7.977009e+09
dtype: float64
Медиана оборота: 99783710.0


In [9]:
import pandas as pd
from sqlalchemy import text
import numpy as np

with engine.connect() as conn:
    customers = pd.read_sql(text("SELECT customer_id, signup_date, tariff_code FROM seg_customers"), conn)

    txn_monthly = pd.read_sql(text("""
        SELECT customer_id, date_trunc('month', txn_ts)::date AS month_start,
               SUM(CASE WHEN direction='out' THEN amount_uzs ELSE 0 END) AS spend_out,
               SUM(CASE WHEN direction='in' THEN amount_uzs ELSE 0 END) AS spend_in,
               COUNT(*) AS txn_cnt,
               SUM(CASE WHEN channel='pos' THEN amount_uzs ELSE 0 END) AS pos_amt,
               SUM(CASE WHEN channel='ecom' THEN amount_uzs ELSE 0 END) AS ecom_amt,
               SUM(CASE WHEN channel='atm' THEN amount_uzs ELSE 0 END) AS atm_amt,
               SUM(CASE WHEN channel='p2p' THEN amount_uzs ELSE 0 END) AS p2p_amt
        FROM seg_transactions
        GROUP BY customer_id, date_trunc('month', txn_ts)::date
    """), conn)

    balances = pd.read_sql(text("SELECT customer_id, month_start, avg_balance_uzs FROM seg_balances"), conn)
    revenue = pd.read_sql(text("SELECT customer_id, month_start, net_revenue_uzs FROM seg_revenue_monthly"), conn)
    products = pd.read_sql(text("SELECT customer_id, COUNT(*) AS n_products FROM seg_products GROUP BY customer_id"), conn)

for df in [txn_monthly, balances, revenue]:
    df['month_start'] = pd.to_datetime(df['month_start'])

print("customers:", customers.shape)
print("txn_monthly:", txn_monthly.shape)
print("balances:", balances.shape)
print("revenue:", revenue.shape)
print("products:", products.shape)

# сохраним в pickle, чтобы не дёргать базу каждый раз
customers.to_pickle('customers.pkl')
txn_monthly.to_pickle('txn_monthly.pkl')
balances.to_pickle('balances.pkl')
revenue.to_pickle('revenue.pkl')
products.to_pickle('products.pkl')
print("\nсохранено")

customers: (20000, 3)
txn_monthly: (410100, 9)
balances: (427598, 3)
revenue: (427598, 3)
products: (15844, 2)

сохранено


In [10]:
import pandas as pd
import numpy as np

customers = pd.read_pickle('customers.pkl')
txn_monthly = pd.read_pickle('txn_monthly.pkl')
balances = pd.read_pickle('balances.pkl')
revenue = pd.read_pickle('revenue.pkl')
products = pd.read_pickle('products.pkl')

# --- выделяем "спящих" (нет ни одной транзакции) ---
active_ids = set(txn_monthly['customer_id'].unique())
all_ids = set(customers['customer_id'].unique())
dormant_ids = all_ids - active_ids
print("Спящих (без транзакций):", len(dormant_ids))

# --- для активных: последний месяц и окно последних 6 месяцев ---
last_month = txn_monthly.groupby('customer_id')['month_start'].max().rename('last_month')
txn = txn_monthly.merge(last_month, on='customer_id')
txn['months_from_last'] = (txn['last_month'].dt.year - txn['month_start'].dt.year) * 12 + \
                           (txn['last_month'].dt.month - txn['month_start'].dt.month)
window = txn[txn['months_from_last'] < 6].copy()  # последние 6 месяцев активности

# агрегаты по окну
agg = window.groupby('customer_id').agg(
    n_months_window=('month_start', 'nunique'),
    avg_spend_out=('spend_out', 'mean'),
    avg_spend_in=('spend_in', 'mean'),
    avg_txn_cnt=('txn_cnt', 'mean'),
    total_pos=('pos_amt', 'sum'),
    total_ecom=('ecom_amt', 'sum'),
    total_atm=('atm_amt', 'sum'),
    total_p2p=('p2p_amt', 'sum'),
).reset_index()

total_channel = agg[['total_pos','total_ecom','total_atm','total_p2p']].sum(axis=1).replace(0, np.nan)
agg['share_pos'] = agg['total_pos'] / total_channel
agg['share_ecom'] = agg['total_ecom'] / total_channel
agg['share_atm'] = agg['total_atm'] / total_channel
agg['share_p2p'] = agg['total_p2p'] / total_channel
agg[['share_pos','share_ecom','share_atm','share_p2p']] = agg[['share_pos','share_ecom','share_atm','share_p2p']].fillna(0)

# --- деньги для банка: баланс и доходность за последние 6 мес относительно last_month ---
bal = balances.merge(last_month, on='customer_id')
bal['months_from_last'] = (bal['last_month'].dt.year - bal['month_start'].dt.year) * 12 + \
                           (bal['last_month'].dt.month - bal['month_start'].dt.month)
bal_window = bal[(bal['months_from_last'] >= 0) & (bal['months_from_last'] < 6)]
bal_agg = bal_window.groupby('customer_id')['avg_balance_uzs'].mean().rename('avg_balance_6m').reset_index()

rev = revenue.merge(last_month, on='customer_id')
rev['months_from_last'] = (rev['last_month'].dt.year - rev['month_start'].dt.year) * 12 + \
                           (rev['last_month'].dt.month - rev['month_start'].dt.month)
rev_window = rev[(rev['months_from_last'] >= 0) & (rev['months_from_last'] < 6)]
rev_agg = rev_window.groupby('customer_id')['net_revenue_uzs'].mean().rename('avg_net_revenue_6m').reset_index()

# --- стаж и тренд (на полной истории клиента) ---
full_hist = txn_monthly.groupby('customer_id')['month_start'].agg(['min','max','nunique']).reset_index()
full_hist.columns = ['customer_id','first_month','last_month_full','n_months_total']

pivot_spend = txn_monthly.pivot(index='customer_id', columns='month_start', values='spend_out').fillna(0)
def trend_ratio(row):
    vals = row.values
    n = len(vals)
    if n < 4:
        return np.nan
    half = n // 2
    first = vals[:half].mean()
    second = vals[half:].mean()
    return (second + 1) / (first + 1)
trend = pivot_spend.apply(trend_ratio, axis=1).rename('trend_ratio').reset_index()

# --- продукты ---
prod = products.rename(columns={'n_products':'n_products'})

# --- собираем всё вместе ---
features = customers[customers['customer_id'].isin(active_ids)].copy()
features = features.merge(agg, on='customer_id', how='left')
features = features.merge(bal_agg, on='customer_id', how='left')
features = features.merge(rev_agg, on='customer_id', how='left')
features = features.merge(full_hist[['customer_id','n_months_total']], on='customer_id', how='left')
features = features.merge(trend, on='customer_id', how='left')
features = features.merge(prod, on='customer_id', how='left')
features['n_products'] = features['n_products'].fillna(0)

print("features shape:", features.shape)
print(features.isna().sum())
features.to_pickle('features_raw.pkl')
print("\nсохранено features_raw.pkl")
print(features.describe().T)

Спящих (без транзакций): 245
features shape: (19755, 20)
customer_id           0
signup_date           0
tariff_code           0
n_months_window       0
avg_spend_out         0
avg_spend_in          0
avg_txn_cnt           0
total_pos             0
total_ecom            0
total_atm             0
total_p2p             0
share_pos             0
share_ecom            0
share_atm             0
share_p2p             0
avg_balance_6m        0
avg_net_revenue_6m    0
n_months_total        0
trend_ratio           0
n_products            0
dtype: int64

сохранено features_raw.pkl
                      count          mean           std           min  \
customer_id         19755.0  9.996054e+03  5.775498e+03  1.000000e+00   
n_months_window     19755.0  5.807644e+00  7.053164e-01  1.000000e+00   
avg_spend_out       19755.0  1.193496e+07  7.282866e+07  0.000000e+00   
avg_spend_in        19755.0  8.665923e+06  5.675158e+07  0.000000e+00   
avg_txn_cnt         19755.0  2.921797e+01  3.153251e+01  

In [11]:
import pandas as pd
import numpy as np

features = pd.read_pickle('features_raw.pkl')

# --- 1. Чиним trend_ratio: пересчитываем через логарифм отношения с разумным сглаживанием,
# и клипуем экстремальные случаи (когда первая половина истории почти нулевая - это шум, не сигнал) ---
features['trend_ratio_clipped'] = features['trend_ratio'].clip(upper=10, lower=0.1)
features['log_trend'] = np.log(features['trend_ratio_clipped'])

# --- 2. Winsorize денежные признаки на 99.5 перцентиле, затем логарифмируем ---
money_cols = ['avg_spend_out', 'avg_spend_in', 'total_pos', 'total_ecom', 'total_atm',
              'total_p2p', 'avg_balance_6m', 'avg_net_revenue_6m']

for col in money_cols:
    cap = features[col].quantile(0.995)
    features[col + '_wz'] = features[col].clip(upper=cap)
    features['log_' + col] = np.log1p(features[col + '_wz'])

print("До/после winsorize+log (avg_spend_out):")
print(features[['avg_spend_out', 'avg_spend_out_wz', 'log_avg_spend_out']].describe())

print("\nlog_trend describe:")
print(features['log_trend'].describe())

print("\nОстальные log-признаки:")
log_cols = ['log_' + c for c in money_cols]
print(features[log_cols].describe().T[['mean','std','min','max']])

features.to_pickle('features_clean.pkl')
print("\nсохранено features_clean.pkl")

До/после winsorize+log (avg_spend_out):
       avg_spend_out  avg_spend_out_wz  log_avg_spend_out
count   1.975500e+04      1.975500e+04       19755.000000
mean    1.193496e+07      1.053292e+07          15.221274
std     7.282866e+07      1.798217e+07           1.643988
min     0.000000e+00      0.000000e+00           0.000000
25%     2.087547e+06      2.087547e+06          14.551501
50%     4.732603e+06      4.732603e+06          15.369986
75%     1.026759e+07      1.026759e+07          16.144503
max     8.085746e+09      1.217020e+08          18.617086

log_trend describe:
count    19755.000000
mean         0.384136
std          0.898252
min         -2.302585
25%         -0.088555
50%          0.076139
75%          0.434920
max          2.302585
Name: log_trend, dtype: float64

Остальные log-признаки:
                             mean       std        min        max
log_avg_spend_out       15.221274  1.643988   0.000000  18.617086
log_avg_spend_in        10.121001  6.887248   0.0000

In [12]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler

features = pd.read_pickle('features_clean.pkl')

features['log_avg_txn_cnt'] = np.log1p(features['avg_txn_cnt'])

feature_cols = [
    'log_avg_spend_out', 'log_avg_txn_cnt', 'log_avg_balance_6m', 'log_avg_net_revenue_6m',
    'share_pos', 'share_ecom', 'share_atm', 'share_p2p',
    'n_months_total', 'log_trend', 'n_products'
]

X = features[feature_cols].copy()
print("NaN по признакам:")
print(X.isna().sum())

# проверим корреляции между признаками - нет ли явного дублирования
corr = X.corr()
print("\nКорреляционная матрица:")
print(corr.round(2))

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols, index=features['customer_id'])

print("\nX_scaled describe:")
print(X_scaled_df.describe().T[['mean','std','min','max']])

X_scaled_df.to_pickle('X_scaled.pkl')
features.to_pickle('features_final.pkl')
print("\nсохранено")

NaN по признакам:
log_avg_spend_out         0
log_avg_txn_cnt           0
log_avg_balance_6m        0
log_avg_net_revenue_6m    0
share_pos                 0
share_ecom                0
share_atm                 0
share_p2p                 0
n_months_total            0
log_trend                 0
n_products                0
dtype: int64

Корреляционная матрица:
                        log_avg_spend_out  log_avg_txn_cnt  \
log_avg_spend_out                    1.00             0.80   
log_avg_txn_cnt                      0.80             1.00   
log_avg_balance_6m                   0.52             0.29   
log_avg_net_revenue_6m               0.87             0.73   
share_pos                           -0.30            -0.42   
share_ecom                           0.20             0.31   
share_atm                           -0.27            -0.34   
share_p2p                            0.27             0.34   
n_months_total                       0.23             0.18   
log_trend       

In [13]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

X_scaled_df = pd.read_pickle('X_scaled.pkl')
X = X_scaled_df.values

results = []
for k in range(3, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    sil = silhouette_score(X, labels, sample_size=5000, random_state=42)
    db = davies_bouldin_score(X, labels)
    sizes = pd.Series(labels).value_counts().sort_index().tolist()
    results.append({'k': k, 'inertia': km.inertia_, 'silhouette': sil, 'davies_bouldin': db, 'cluster_sizes': sizes})
    print(f"k={k}: inertia={km.inertia_:.0f}, silhouette={sil:.3f}, davies_bouldin={db:.3f}, sizes={sizes}")

results_df = pd.DataFrame(results)
results_df.to_pickle('k_selection_results.pkl')

/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: divide by zero encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: overflow encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/alishersabirov

k=3: inertia=117828, silhouette=0.256, davies_bouldin=1.348, sizes=[9436, 6970, 3349]
k=4: inertia=106397, silhouette=0.226, davies_bouldin=1.473, sizes=[6706, 2043, 3310, 7696]


/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklear

k=5: inertia=98654, silhouette=0.203, davies_bouldin=1.581, sizes=[4356, 1890, 3262, 4051, 6196]


/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklear

k=6: inertia=92551, silhouette=0.176, davies_bouldin=1.713, sizes=[1840, 1438, 1968, 6170, 4324, 4015]
k=7: inertia=87019, silhouette=0.174, davies_bouldin=1.655, sizes=[1593, 3269, 1962, 1421, 3804, 4399, 3307]


/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklear

k=8: inertia=82639, silhouette=0.176, davies_bouldin=1.550, sizes=[3089, 1261, 3132, 4273, 1521, 1230, 3641, 1608]


/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklear

In [14]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

X_scaled_df = pd.read_pickle('X_scaled.pkl')
X = X_scaled_df.values

print("inf в X:", np.isinf(X).sum())
print("nan в X:", np.isnan(X).sum())
print("Экстремальные значения (|z|>8):")
extreme_mask = np.abs(X) > 8
print(pd.Series(extreme_mask.sum(axis=0), index=X_scaled_df.columns))

# клипуем экстремальные z-score на +-5, чтобы не давать редким выбросам взрывать дистанции
X_clipped = np.clip(X, -5, 5)
X_scaled_df_clipped = pd.DataFrame(X_clipped, columns=X_scaled_df.columns, index=X_scaled_df.index)
X_scaled_df_clipped.to_pickle('X_scaled_clipped.pkl')

results = []
for k in range(3, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_clipped)
    sil = silhouette_score(X_clipped, labels, sample_size=5000, random_state=42)
    db = davies_bouldin_score(X_clipped, labels)
    sizes = pd.Series(labels).value_counts().sort_index().tolist()
    results.append({'k': k, 'inertia': km.inertia_, 'silhouette': sil, 'davies_bouldin': db, 'cluster_sizes': sizes})

results_df = pd.DataFrame(results)
pd.set_option('display.width', 120)
print("\n" + results_df.to_string(index=False))
results_df.to_pickle('k_selection_results_clean.pkl')

inf в X: 0
nan в X: 0
Экстремальные значения (|z|>8):
log_avg_spend_out         45
log_avg_txn_cnt            0
log_avg_balance_6m         0
log_avg_net_revenue_6m     0
share_pos                  0
share_ecom                 0
share_atm                  0
share_p2p                  0
n_months_total             0
log_trend                  0
n_products                 0
dtype: int64


/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: divide by zero encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: overflow encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/alishersabirov


 k       inertia  silhouette  davies_bouldin                                    cluster_sizes
 3 114407.949866    0.254318        1.349963                               [3355, 7222, 9178]
 4 103505.954548    0.226356        1.475545                         [2105, 6667, 3316, 7667]
 5  95807.687669    0.205209        1.580630                   [6186, 3269, 1926, 4033, 4341]
 6  89725.779049    0.177673        1.716676             [4315, 1452, 1888, 6151, 1960, 3989]
 7  84236.849867    0.175628        1.658680       [1671, 4364, 1427, 3284, 3262, 1956, 3791]
 8  79873.423825    0.178123        1.550858 [1217, 1663, 3075, 4247, 1257, 1539, 3124, 3633]


/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklear

In [15]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans

X_scaled_df_clipped = pd.read_pickle('X_scaled_clipped.pkl')
features = pd.read_pickle('features_final.pkl')

X = X_scaled_df_clipped.values
raw_cols = ['avg_spend_out', 'avg_balance_6m', 'avg_net_revenue_6m', 'avg_txn_cnt',
            'share_pos', 'share_ecom', 'share_atm', 'share_p2p',
            'n_months_total', 'n_products']

profiles = {}
for k in [3, 4, 5, 6]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    features[f'cluster_k{k}'] = labels
    profile = features.groupby(f'cluster_k{k}')[raw_cols].median()
    profile['n_clients'] = features.groupby(f'cluster_k{k}').size()
    profile['pct'] = (profile['n_clients'] / len(features) * 100).round(1)
    profiles[k] = profile
    print(f"\n{'='*80}\nk={k}\n{'='*80}")
    pd.set_option('display.width', 160)
    pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
    print(profile.to_string())

features.to_pickle('features_with_clusters.pkl')

/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: divide by zero encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: overflow encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/alishersabirov


k=3
            avg_spend_out  avg_balance_6m  avg_net_revenue_6m  avg_txn_cnt  share_pos  share_ecom  share_atm  share_p2p  n_months_total  n_products  n_clients   pct
cluster_k3                                                                                                                                                          
0            4,872,826.00    1,606,049.50           41,770.33        16.33       0.38        0.12       0.17       0.12           10.00        1.00       3355 17.00
1            1,874,706.50      751,029.25           13,758.25        10.00       0.47        0.06       0.29       0.00           24.00        1.00       7222 36.60
2            9,089,966.50    3,436,351.08           93,828.83        35.37       0.32        0.20       0.12       0.21           24.00        2.00       9178 46.50

k=4
            avg_spend_out  avg_balance_6m  avg_net_revenue_6m  avg_txn_cnt  share_pos  share_ecom  share_atm  share_p2p  n_months_total  n_products  n_clients   pct


/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: divide by zero encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: overflow encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/alishersabirov

In [16]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

features = pd.read_pickle('features_final.pkl')

behavior_cols = [
    'log_avg_spend_out', 'log_avg_txn_cnt', 'log_avg_balance_6m', 'log_avg_net_revenue_6m',
    'share_pos', 'share_ecom', 'share_atm', 'share_p2p', 'n_products'
]

from sklearn.preprocessing import RobustScaler
X2 = features[behavior_cols].copy()
scaler2 = RobustScaler()
X2_scaled = scaler2.fit_transform(X2)
X2_scaled = np.clip(X2_scaled, -5, 5)

X2_scaled_df = pd.DataFrame(X2_scaled, columns=behavior_cols, index=features['customer_id'])
X2_scaled_df.to_pickle('X2_scaled.pkl')

results = []
for k in range(3, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X2_scaled)
    sil = silhouette_score(X2_scaled, labels, sample_size=5000, random_state=42)
    db = davies_bouldin_score(X2_scaled, labels)
    sizes = pd.Series(labels).value_counts().sort_index().tolist()
    results.append({'k': k, 'inertia': km.inertia_, 'silhouette': sil, 'davies_bouldin': db, 'cluster_sizes': sizes})

results_df = pd.DataFrame(results)
pd.set_option('display.width', 160)
print(results_df.to_string(index=False))

# профили по k=4,5,6 чтобы проверить - ушёл ли tenure-артефакт
raw_cols = ['avg_spend_out', 'avg_balance_6m', 'avg_net_revenue_6m', 'avg_txn_cnt',
            'share_pos', 'share_ecom', 'share_atm', 'share_p2p', 'n_months_total', 'n_products']

for k in [4, 5, 6]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X2_scaled)
    features[f'cluster2_k{k}'] = labels
    profile = features.groupby(f'cluster2_k{k}')[raw_cols].median()
    profile['n_clients'] = features.groupby(f'cluster2_k{k}').size()
    profile['pct'] = (profile['n_clients'] / len(features) * 100).round(1)
    print(f"\n{'='*80}\nk={k}\n{'='*80}")
    pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
    print(profile.to_string())

features.to_pickle('features_with_clusters2.pkl')

/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: divide by zero encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: overflow encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/alishersabirov

 k   inertia  silhouette  davies_bouldin                                   cluster_sizes
 3 70,460.90        0.20            1.62                              [6018, 7292, 6445]
 4 63,040.58        0.20            1.60                        [7285, 4870, 2496, 5104]
 5 56,525.40        0.21            1.49                  [5145, 4415, 4089, 2220, 3886]
 6 51,726.23        0.22            1.47            [2123, 2165, 3614, 4810, 3282, 3761]
 7 48,123.80        0.22            1.43      [2114, 3389, 2133, 2346, 4636, 1933, 3204]
 8 45,255.16        0.22            1.41 [2143, 4484, 923, 1900, 3261, 1542, 2302, 3200]

k=4
             avg_spend_out  avg_balance_6m  avg_net_revenue_6m  avg_txn_cnt  share_pos  share_ecom  share_atm  share_p2p  n_months_total  n_products  n_clients   pct
cluster2_k4                                                                                                                                                          
0             2,933,624.33    1,060,865.

/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: divide by zero encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: overflow encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/alishersabirov

In [17]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score

X2_scaled_df = pd.read_pickle('X2_scaled.pkl')
features = pd.read_pickle('features_with_clusters2.pkl')
X = X2_scaled_df.values

# KMeans (базовый, уже есть labels в features['cluster2_k4'])
km_labels = features['cluster2_k4'].values

# GaussianMixture
gmm = GaussianMixture(n_components=4, random_state=42, n_init=5)
gmm_labels = gmm.fit_predict(X)

# Agglomerative (Ward) - на подвыборке, т.к. O(n^2) память для 19755 точек может быть тяжёлой
np.random.seed(42)
sample_idx = np.random.choice(len(X), size=5000, replace=False)
agg = AgglomerativeClustering(n_clusters=4, linkage='ward')
agg_labels_sample = agg.fit_predict(X[sample_idx])

print("ARI: KMeans vs GaussianMixture:", adjusted_rand_score(km_labels, gmm_labels))
print("ARI: KMeans vs Agglomerative (на подвыборке 5000):",
      adjusted_rand_score(km_labels[sample_idx], agg_labels_sample))

# профиль GMM
raw_cols = ['avg_spend_out', 'avg_balance_6m', 'avg_net_revenue_6m', 'avg_txn_cnt',
            'share_pos', 'share_ecom', 'share_atm', 'share_p2p', 'n_months_total', 'n_products']
features['cluster_gmm'] = gmm_labels
profile_gmm = features.groupby('cluster_gmm')[raw_cols].median()
profile_gmm['n_clients'] = features.groupby('cluster_gmm').size()
profile_gmm['pct'] = (profile_gmm['n_clients'] / len(features) * 100).round(1)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print("\nПрофиль GaussianMixture (k=4):")
print(profile_gmm.to_string())

features.to_pickle('features_with_clusters3.pkl')

/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: divide by zero encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: overflow encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/alishersabirov

ARI: KMeans vs GaussianMixture: 0.10242436147308906
ARI: KMeans vs Agglomerative (на подвыборке 5000): 0.4343171031710319

Профиль GaussianMixture (k=4):
             avg_spend_out  avg_balance_6m  avg_net_revenue_6m  avg_txn_cnt  share_pos  share_ecom  share_atm  share_p2p  n_months_total  n_products  n_clients   pct
cluster_gmm                                                                                                                                                          
0             3,920,693.92    2,873,232.17           35,703.75        14.50       0.36        0.09       0.19       0.21           24.00        1.00       4484 22.70
1             1,597,059.58      851,856.67           11,692.08         6.83       0.55        0.09       0.29       0.00           23.00        1.00       5632 28.50
2             3,180,067.50   15,000,369.50           99,081.67         7.67       0.36        0.09       0.17       0.24           24.00        2.00       1249  6.30
3             8,

In [18]:
import pandas as pd
import numpy as np
from sklearn.metrics import adjusted_rand_score

features = pd.read_pickle('features_with_clusters3.pkl')

# квартильный бейзлайн по обороту (используем total spend за окно как прокси оборота)
features['quartile_baseline'] = pd.qcut(features['avg_spend_out'], q=4, labels=False, duplicates='drop')

# 1. ARI между нашей типологией и квартильным бейзлайном - насколько они вообще разные разбиения
ari_vs_quartile = adjusted_rand_score(features['cluster2_k4'], features['quartile_baseline'])
print("ARI: наша типология (k=4) vs квартили по обороту:", ari_vs_quartile)

# 2. Внутригрупповая однородность: сравним дисперсию признаков, НЕ связанных с оборотом напрямую,
# внутри кластеров нашей типологии vs внутри квартилей оборота
compare_cols = ['share_pos', 'share_ecom', 'share_atm', 'share_p2p', 'n_products', 'avg_balance_6m']

def within_group_variance_ratio(df, group_col, cols):
    total_var = df[cols].var()
    within_var = df.groupby(group_col)[cols].var().mean()  # средняя внутригрупповая дисперсия
    return (within_var / total_var).mean()  # чем меньше, тем однороднее группы

ratio_ours = within_group_variance_ratio(features, 'cluster2_k4', compare_cols)
ratio_quartile = within_group_variance_ratio(features, 'quartile_baseline', compare_cols)
print(f"\nСредняя доля необъяснённой (внутригрупповой) дисперсии по каналам/продуктам/балансу:")
print(f"  Наша типология: {ratio_ours:.3f}  (меньше = однороднее группы)")
print(f"  Квартили по обороту: {ratio_quartile:.3f}")

# 3. Профиль квартилей для сравнения "глазами"
raw_cols = ['avg_spend_out', 'avg_balance_6m', 'avg_net_revenue_6m', 'avg_txn_cnt',
            'share_pos', 'share_ecom', 'share_atm', 'share_p2p', 'n_products']
profile_q = features.groupby('quartile_baseline')[raw_cols].median()
profile_q['n_clients'] = features.groupby('quartile_baseline').size()
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print("\nПрофиль квартилей по обороту:")
print(profile_q.to_string())

features.to_pickle('features_with_clusters4.pkl')

ARI: наша типология (k=4) vs квартили по обороту: 0.22518997582806966

Средняя доля необъяснённой (внутригрупповой) дисперсии по каналам/продуктам/балансу:
  Наша типология: 0.802  (меньше = однороднее группы)
  Квартили по обороту: 0.895

Профиль квартилей по обороту:
                   avg_spend_out  avg_balance_6m  avg_net_revenue_6m  avg_txn_cnt  share_pos  share_ecom  share_atm  share_p2p  n_products  n_clients
quartile_baseline                                                                                                                                    
0                     778,852.67      616,104.33            7,412.50         5.00       0.49        0.05       0.28       0.00        1.00       4939
1                   3,315,005.67    1,325,954.83           25,336.00        14.67       0.40        0.08       0.22       0.12        1.00       4939
2                   6,863,892.83    2,084,652.67           60,000.92        24.67       0.36        0.21       0.14       0.14    

In [19]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import adjusted_rand_score

features = pd.read_pickle('features_with_clusters4.pkl')
X2_scaled_df = pd.read_pickle('X2_scaled.pkl')
X_base = X2_scaled_df.values
base_labels = features['cluster2_k4'].values
customer_ids = features['customer_id'].values

behavior_cols = [
    'log_avg_spend_out', 'log_avg_txn_cnt', 'log_avg_balance_6m', 'log_avg_net_revenue_6m',
    'share_pos', 'share_ecom', 'share_atm', 'share_p2p', 'n_products'
]

stability_results = []

# --- 1. Другой random_state (несколько прогонов) ---
for rs in [0, 7, 99]:
    km = KMeans(n_clusters=4, random_state=rs, n_init=10)
    labels = km.fit_predict(X_base)
    ari = adjusted_rand_score(base_labels, labels)
    stability_results.append({'test': f'random_state={rs}', 'ARI': ari})

# --- 2. Половина выборки ---
np.random.seed(1)
half_idx = np.random.choice(len(X_base), size=len(X_base)//2, replace=False)
km_half = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_half = km_half.fit_predict(X_base[half_idx])
ari_half = adjusted_rand_score(base_labels[half_idx], labels_half)
stability_results.append({'test': 'половина выборки', 'ARI': ari_half})

# --- 3. Без последнего месяца данных (пересчитываем окно: последние 6 мес, сдвинутые на 1 назад) ---
txn_monthly = pd.read_pickle('txn_monthly.pkl')
last_month_all = txn_monthly.groupby('customer_id')['month_start'].max().rename('last_month')
txn2 = txn_monthly.merge(last_month_all, on='customer_id')
# исключаем последний месяц каждого клиента, затем берём окно 6 мес от нового "последнего"
txn2_excl = txn2[txn2['month_start'] < txn2['last_month']].copy()
new_last = txn2_excl.groupby('customer_id')['month_start'].max().rename('new_last_month')
txn2_excl = txn2_excl.merge(new_last, on='customer_id')
txn2_excl['months_from_last'] = (txn2_excl['new_last_month'].dt.year - txn2_excl['month_start'].dt.year)*12 + \
                                 (txn2_excl['new_last_month'].dt.month - txn2_excl['month_start'].dt.month)
window2 = txn2_excl[txn2_excl['months_from_last'] < 6]

agg2 = window2.groupby('customer_id').agg(
    avg_spend_out=('spend_out','mean'), avg_txn_cnt=('txn_cnt','mean'),
    total_pos=('pos_amt','sum'), total_ecom=('ecom_amt','sum'),
    total_atm=('atm_amt','sum'), total_p2p=('p2p_amt','sum')
).reset_index()
tot_ch = agg2[['total_pos','total_ecom','total_atm','total_p2p']].sum(axis=1).replace(0,np.nan)
agg2['share_pos'] = (agg2['total_pos']/tot_ch).fillna(0)
agg2['share_ecom'] = (agg2['total_ecom']/tot_ch).fillna(0)
agg2['share_atm'] = (agg2['total_atm']/tot_ch).fillna(0)
agg2['share_p2p'] = (agg2['total_p2p']/tot_ch).fillna(0)
agg2['log_avg_spend_out'] = np.log1p(agg2['avg_spend_out'].clip(upper=agg2['avg_spend_out'].quantile(0.995)))
agg2['log_avg_txn_cnt'] = np.log1p(agg2['avg_txn_cnt'])

merged2 = features[['customer_id','avg_balance_6m','avg_net_revenue_6m','n_products']].merge(agg2, on='customer_id', how='inner')
merged2['log_avg_balance_6m'] = np.log1p(merged2['avg_balance_6m'])
merged2['log_avg_net_revenue_6m'] = np.log1p(merged2['avg_net_revenue_6m'].clip(lower=0))

X3 = merged2[behavior_cols].values
X3_scaled = np.clip(RobustScaler().fit_transform(X3), -5, 5)
km3 = KMeans(n_clusters=4, random_state=42, n_init=10)
labels3 = km3.fit_predict(X3_scaled)

common_ids = merged2['customer_id'].values
base_labels_map = dict(zip(customer_ids, base_labels))
base_labels_common = np.array([base_labels_map[cid] for cid in common_ids])
ari_no_last_month = adjusted_rand_score(base_labels_common, labels3)
stability_results.append({'test': 'без последнего месяца', 'ARI': ari_no_last_month})

# --- 4. Изменённый набор признаков (убираем n_products и share_p2p) ---
alt_cols = ['log_avg_spend_out', 'log_avg_txn_cnt', 'log_avg_balance_6m', 'log_avg_net_revenue_6m',
            'share_pos', 'share_ecom', 'share_atm']
X4 = features[alt_cols].values
X4_scaled = np.clip(RobustScaler().fit_transform(X4), -5, 5)
km4 = KMeans(n_clusters=4, random_state=42, n_init=10)
labels4 = km4.fit_predict(X4_scaled)
ari_alt_features = adjusted_rand_score(base_labels, labels4)
stability_results.append({'test': 'изменённый набор признаков', 'ARI': ari_alt_features})

stability_df = pd.DataFrame(stability_results)
print(stability_df.to_string(index=False))
stability_df.to_pickle('stability_results.pkl')

/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: divide by zero encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: overflow encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/alishersabirov

                      test  ARI
            random_state=0 1.00
            random_state=7 1.00
           random_state=99 1.00
          половина выборки 0.98
     без последнего месяца 0.94
изменённый набор признаков 0.62


/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: divide by zero encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: overflow encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/alishersabirov

In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score

features = pd.read_pickle('features_with_clusters4.pkl')
X2_scaled_df = pd.read_pickle('X2_scaled.pkl')

X = X2_scaled_df.values
y = features['cluster2_k4'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

knn = KNeighborsClassifier(n_neighbors=15)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)

print("Accuracy правила отнесения нового клиента (kNN, k=15):", accuracy_score(y_test, y_pred))
print("\nClassification report:")
print(classification_report(y_test, y_pred))

# также проверим ближайший центроид (проще для продакшена - не нужно хранить весь train)
centroids = pd.DataFrame(X_train, columns=X2_scaled_df.columns)
centroids['cluster'] = y_train
centroid_means = centroids.groupby('cluster').mean().values

def assign_to_nearest_centroid(x, centroids):
    dists = np.linalg.norm(centroids - x, axis=1)
    return np.argmin(dists)

y_pred_centroid = np.array([assign_to_nearest_centroid(x, centroid_means) for x in X_test])
print("\nAccuracy правила через ближайший центроид:", accuracy_score(y_test, y_pred_centroid))

Accuracy правила отнесения нового клиента (kNN, k=15): 0.9749430523917996

Classification report:
              precision    recall  f1-score   support

           0       0.96      0.98      0.97      1457
           1       0.98      0.96      0.97       974
           2       0.99      0.95      0.97       499
           3       0.99      0.98      0.98      1021

    accuracy                           0.97      3951
   macro avg       0.98      0.97      0.97      3951
weighted avg       0.98      0.97      0.97      3951


Accuracy правила через ближайший центроид: 0.9989875980764363


In [21]:
import pandas as pd
import numpy as np
from sqlalchemy import text

features = pd.read_pickle('features_with_clusters4.pkl')

with engine.connect() as conn:
    tariff = pd.read_sql(text("SELECT * FROM seg_tariff_current"), conn)
print("Текущий единый тариф:")
print(tariff.to_string(index=False))

# итоговая сводка по кластерам: размер, деньги сейчас, потенциал
raw_cols = ['avg_spend_out', 'avg_balance_6m', 'avg_net_revenue_6m', 'avg_txn_cnt',
            'share_pos', 'share_ecom', 'share_atm', 'share_p2p', 'n_products']

names = {
    0: 'Наличностники',
    1: 'Онлайн-активные',
    2: 'Спящие',
    3: 'Премиум мульти-продуктовые'
}
features['segment_name'] = features['cluster2_k4'].map(names)

summary = features.groupby('segment_name').agg(
    n_clients=('customer_id', 'count'),
    total_monthly_revenue=('avg_net_revenue_6m', 'sum'),
    avg_revenue_per_client=('avg_net_revenue_6m', 'mean'),
    median_balance=('avg_balance_6m', 'median'),
    median_spend=('avg_spend_out', 'median'),
    avg_products=('n_products', 'mean'),
).round(0)
summary['pct_of_base'] = (summary['n_clients'] / len(features) * 100).round(1)
summary['pct_of_revenue'] = (summary['total_monthly_revenue'] / summary['total_monthly_revenue'].sum() * 100).round(1)

pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f'{x:,.0f}')
print("\nСводка по сегментам:")
print(summary.to_string())

# добавляем "спящих" (245 клиентов без транзакций) отдельной строкой для полноты картины
print("\nСправочно: 245 клиентов без единой транзакции (отдельная категория, не в кластеризации)")

features.to_pickle('features_final_named.pkl')
summary.to_pickle('segment_summary.pkl')

Текущий единый тариф:
tariff_code  monthly_fee_uzs  atm_free_limit_uzs  transfer_fee_pct                                                                                       description
   standard            15000             2000000              0.50 Единый тариф для всех клиентов: абонентская плата, лимит бесплатного снятия, комиссия за переводы

Сводка по сегментам:
                            n_clients  total_monthly_revenue  avg_revenue_per_client  median_balance  median_spend  avg_products  pct_of_base  pct_of_revenue
segment_name                                                                                                                                                 
Наличностники                    7285            189,328,121                  25,989       1,060,865     2,933,624             1           37               9
Онлайн-активные                  4870            594,566,308                 122,088       2,389,267     8,773,899             1           25           

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

features = pd.read_pickle('features_final_named.pkl')
X2_scaled_df = pd.read_pickle('X2_scaled.pkl')
stability_df = pd.read_pickle('stability_results.pkl')
summary = pd.read_pickle('segment_summary.pkl')

colors = {'Наличностники':'#4C72B0','Онлайн-активные':'#DD8452','Спящие':'#999999','Премиум мульти-продуктовые':'#55A868'}

# --- 1. PCA scatter ---
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X2_scaled_df.values)
features['pca1'] = X_pca[:,0]
features['pca2'] = X_pca[:,1]

fig, ax = plt.subplots(figsize=(9,7))
for name, grp in features.groupby('segment_name'):
    ax.scatter(grp['pca1'], grp['pca2'], s=6, alpha=0.4, label=name, color=colors.get(name))
ax.set_title(f'Кластеры клиентов в проекции PCA (объяснено {pca.explained_variance_ratio_.sum()*100:.0f}% дисперсии)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.legend(markerscale=4)
plt.tight_layout()
plt.savefig('pca_scatter.png', dpi=150)
plt.close()

# --- 2. Распределения ключевых признаков по сегментам (boxplots) ---
fig, axes = plt.subplots(1, 3, figsize=(16,5))
plot_cols = ['avg_spend_out', 'avg_balance_6m', 'n_products']
titles = ['Траты (мес, сум)', 'Средний баланс (сум)', 'Кол-во продуктов']
order = ['Спящие','Наличностники','Онлайн-активные','Премиум мульти-продуктовые']
for ax, col, title in zip(axes, plot_cols, titles):
    data = [features[features['segment_name']==seg][col].clip(upper=features[col].quantile(0.97)) for seg in order]
    ax.boxplot(data, labels=order, showfliers=False)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('segment_boxplots.png', dpi=150)
plt.close()

# --- 3. Устойчивость (ARI) ---
fig, ax = plt.subplots(figsize=(8,5))
bars = ax.bar(stability_df['test'], stability_df['ARI'], color='#4C72B0')
ax.axhline(0.7, color='red', linestyle='--', linewidth=1, label='порог приемлемой устойчивости (0.7)')
ax.set_ylim(0,1.05)
ax.set_ylabel('ARI')
ax.set_title('Устойчивость типологии (ARI между базовым разбиением и вариациями)')
plt.xticks(rotation=25, ha='right')
ax.legend()
for b in bars:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.02, f'{b.get_height():.2f}', ha='center')
plt.tight_layout()
plt.savefig('stability_chart.png', dpi=150)
plt.close()

# --- 4. Доля базы vs доля дохода по сегментам ---
fig, ax = plt.subplots(figsize=(8,5))
x = np.arange(len(summary))
w = 0.35
ax.bar(x-w/2, summary['pct_of_base'], w, label='% клиентской базы', color='#4C72B0')
ax.bar(x+w/2, summary['pct_of_revenue'], w, label='% доходности банка', color='#DD8452')
ax.set_xticks(x); ax.set_xticklabels(summary.index, rotation=20, ha='right')
ax.set_ylabel('%')
ax.set_title('Доля базы vs доля дохода по сегментам')
ax.legend()
plt.tight_layout()
plt.savefig('revenue_share_chart.png', dpi=150)
plt.close()

print("Графики сохранены: pca_scatter.png, segment_boxplots.png, stability_chart.png, revenue_share_chart.png")
features.to_pickle('features_final_named.pkl')

/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: overflow encountered in matmul
  X_transformed = X @ self.components_.T
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: invalid value encountered in matmul
  X_transformed = X @ self.components_.T
/var/folders/ch/fyp9js953h77f_wvqmdx9n280000gn/T/ipykernel_3720/257001985.py:36: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=order, showfliers=False)
/var/folders/ch/fyp9js953h77f_wvqmdx9n280000gn/T/ipyker

Графики сохранены: pca_scatter.png, segment_boxplots.png, stability_chart.png, revenue_share_chart.png


In [2]:
import pandas as pd
from sqlalchemy import text

with engine.connect() as conn:
    # Q3: у скольких клиентов история короче 6 месяцев (разница между первой и последней операцией < 180 дней)
    q3 = pd.read_sql(text("""
        SELECT COUNT(*) AS n
        FROM (
            SELECT customer_id, MAX(txn_ts)::date - MIN(txn_ts)::date AS history_days
            FROM seg_transactions
            GROUP BY customer_id
        ) t
        WHERE history_days < 180
    """), conn)
    print("Q3 - клиентов с историей короче 180 дней:", q3['n'][0])

    # Q4: у скольких клиентов больше половины ИСХОДЯЩИХ операций - это снятие в банкомате
    q4 = pd.read_sql(text("""
        SELECT COUNT(*) AS n
        FROM (
            SELECT customer_id,
                   SUM(CASE WHEN direction='out' AND channel='atm' THEN 1 ELSE 0 END)::float
                   / NULLIF(SUM(CASE WHEN direction='out' THEN 1 ELSE 0 END), 0) AS atm_share
            FROM seg_transactions
            GROUP BY customer_id
        ) t
        WHERE atm_share > 0.5
    """), conn)
    print("Q4 - клиентов с >50% исходящих операций через ATM:", q4['n'][0])

Q3 - клиентов с историей короче 180 дней: 782
Q4 - клиентов с >50% исходящих операций через ATM: 701


In [5]:
import pandas as pd
from sqlalchemy import text

with engine.connect() as conn:
    # 1. p2p всегда входящий?
    p2p_check = pd.read_sql(text("""
        SELECT direction, COUNT(*) AS n, SUM(amount_uzs) AS total_amt
        FROM seg_transactions
        WHERE channel = 'p2p'
        GROUP BY direction
    """), conn)
    print("P2P по направлению:")
    print(p2p_check.to_string(index=False))

    # 2. cash (mcc_group) ≡ atm (channel)?
    cash_atm_check = pd.read_sql(text("""
        SELECT
            SUM(CASE WHEN mcc_group='cash' AND channel='atm' THEN 1 ELSE 0 END) AS both_cash_atm,
            SUM(CASE WHEN mcc_group='cash' AND channel!='atm' THEN 1 ELSE 0 END) AS cash_not_atm,
            SUM(CASE WHEN mcc_group!='cash' AND channel='atm' THEN 1 ELSE 0 END) AS atm_not_cash,
            COUNT(*) AS total
        FROM seg_transactions
    """), conn)
    print("\nCash vs ATM пересечение:")
    print(cash_atm_check.to_string(index=False))

P2P по направлению:
direction       n    total_amt
       in 1294964 3.558091e+12

Cash vs ATM пересечение:
 both_cash_atm  cash_not_atm  atm_not_cash    total
       2406422             0             0 12136132


In [6]:
import pandas as pd
import numpy as np
from sqlalchemy import text

with engine.connect() as conn:
    txn_monthly_fixed = pd.read_sql(text("""
        SELECT customer_id, date_trunc('month', txn_ts)::date AS month_start,
               SUM(CASE WHEN direction='out' THEN amount_uzs ELSE 0 END) AS spend_out,
               SUM(CASE WHEN direction='in' THEN amount_uzs ELSE 0 END) AS spend_in,
               COUNT(CASE WHEN direction='out' THEN 1 END) AS txn_cnt_out,
               SUM(CASE WHEN direction='out' AND channel='pos' THEN amount_uzs ELSE 0 END) AS pos_out,
               SUM(CASE WHEN direction='out' AND channel='ecom' THEN amount_uzs ELSE 0 END) AS ecom_out,
               SUM(CASE WHEN direction='out' AND channel='atm' THEN amount_uzs ELSE 0 END) AS atm_out
        FROM seg_transactions
        GROUP BY customer_id, date_trunc('month', txn_ts)::date
    """), conn)

txn_monthly_fixed['month_start'] = pd.to_datetime(txn_monthly_fixed['month_start'])
print("Проверка: значения direction='out' по channel='p2p' должны отсутствовать (p2p не входит в pos/ecom/atm - ожидаемо)")
print(txn_monthly_fixed.shape)
print(txn_monthly_fixed.head())

txn_monthly_fixed.to_pickle('txn_monthly_fixed.pkl')
print("\nсохранено txn_monthly_fixed.pkl")

Проверка: значения direction='out' по channel='p2p' должны отсутствовать (p2p не входит в pos/ecom/atm - ожидаемо)
(410100, 8)
   customer_id month_start  spend_out   spend_in  txn_cnt_out    pos_out  \
0            1  2024-03-01  6045823.0        0.0           24  5173726.0   
1            1  2024-04-01  4198729.0  4141305.0           19  2515555.0   
2            1  2024-05-01   712616.0  5574956.0            5   618206.0   
3            1  2024-06-01  4286617.0   143402.0           11  3218372.0   
4            1  2024-07-01  3902765.0  6654261.0           11  3727737.0   

   ecom_out    atm_out  
0  565827.0   306270.0  
1  596079.0  1087095.0  
2       0.0    94410.0  
3  365430.0   702815.0  
4  175028.0        0.0  

сохранено txn_monthly_fixed.pkl


In [7]:
import pandas as pd
import numpy as np

txn = pd.read_pickle('txn_monthly_fixed.pkl')
customers = pd.read_pickle('customers.pkl')
balances = pd.read_pickle('balances.pkl')
revenue = pd.read_pickle('revenue.pkl')
products = pd.read_pickle('products.pkl')

active_ids = set(txn['customer_id'].unique())
all_ids = set(customers['customer_id'].unique())
dormant_ids = all_ids - active_ids
print("Спящих:", len(dormant_ids))

# последний месяц активности (любой, in или out) и окно последних 6 месяцев
last_month = txn.groupby('customer_id')['month_start'].max().rename('last_month')
txn2 = txn.merge(last_month, on='customer_id')
txn2['months_from_last'] = (txn2['last_month'].dt.year - txn2['month_start'].dt.year) * 12 + \
                            (txn2['last_month'].dt.month - txn2['month_start'].dt.month)
window = txn2[txn2['months_from_last'] < 6].copy()

agg = window.groupby('customer_id').agg(
    n_months_window=('month_start', 'nunique'),
    avg_spend_out=('spend_out', 'mean'),
    avg_spend_in=('spend_in', 'mean'),
    avg_txn_cnt_out=('txn_cnt_out', 'mean'),
    total_pos_out=('pos_out', 'sum'),
    total_ecom_out=('ecom_out', 'sum'),
    total_atm_out=('atm_out', 'sum'),
).reset_index()

total_ch = agg[['total_pos_out','total_ecom_out','total_atm_out']].sum(axis=1).replace(0, np.nan)
agg['share_pos'] = (agg['total_pos_out'] / total_ch).fillna(0)
agg['share_ecom'] = (agg['total_ecom_out'] / total_ch).fillna(0)
agg['share_atm'] = (agg['total_atm_out'] / total_ch).fillna(0)
# больше не считаем share_p2p как долю (исходящих p2p не существует)

# баланс и доходность (как раньше)
bal = balances.merge(last_month, on='customer_id')
bal['months_from_last'] = (bal['last_month'].dt.year - bal['month_start'].dt.year) * 12 + \
                           (bal['last_month'].dt.month - bal['month_start'].dt.month)
bal_agg = bal[(bal['months_from_last'] >= 0) & (bal['months_from_last'] < 6)].groupby('customer_id')['avg_balance_uzs'].mean().rename('avg_balance_6m').reset_index()

rev = revenue.merge(last_month, on='customer_id')
rev['months_from_last'] = (rev['last_month'].dt.year - rev['month_start'].dt.year) * 12 + \
                           (rev['last_month'].dt.month - rev['month_start'].dt.month)
rev_agg = rev[(rev['months_from_last'] >= 0) & (rev['months_from_last'] < 6)].groupby('customer_id')['net_revenue_uzs'].mean().rename('avg_net_revenue_6m').reset_index()

# стаж (для профилирования, не для кластеризации)
full_hist = txn.groupby('customer_id')['month_start'].nunique().rename('n_months_total').reset_index()

# продукты
prod = products.rename(columns={'n_products':'n_products'})

# сборка
features2 = customers[customers['customer_id'].isin(active_ids)].copy()
features2 = features2.merge(agg, on='customer_id', how='left')
features2 = features2.merge(bal_agg, on='customer_id', how='left')
features2 = features2.merge(rev_agg, on='customer_id', how='left')
features2 = features2.merge(full_hist, on='customer_id', how='left')
features2 = features2.merge(prod, on='customer_id', how='left')
features2['n_products'] = features2['n_products'].fillna(0)

# логарифмы + winsorize
money_cols = ['avg_spend_out', 'avg_spend_in', 'avg_balance_6m', 'avg_net_revenue_6m']
for col in money_cols:
    cap = features2[col].quantile(0.995)
    features2['log_' + col] = np.log1p(features2[col].clip(upper=cap))

features2['log_avg_txn_cnt_out'] = np.log1p(features2['avg_txn_cnt_out'])
# новый признак вместо share_p2p
features2['in_amt_log'] = features2['log_avg_spend_in']  # уже логарифмирован выше

print("features2 shape:", features2.shape)
print(features2.isna().sum())
features2.to_pickle('features_v2.pkl')
print("\nсохранено features_v2.pkl")
print(features2[['share_pos','share_ecom','share_atm']].describe())

Спящих: 245
features2 shape: (19755, 23)
customer_id               0
signup_date               0
tariff_code               0
n_months_window           0
avg_spend_out             0
avg_spend_in              0
avg_txn_cnt_out           0
total_pos_out             0
total_ecom_out            0
total_atm_out             0
share_pos                 0
share_ecom                0
share_atm                 0
avg_balance_6m            0
avg_net_revenue_6m        0
n_months_total            0
n_products                0
log_avg_spend_out         0
log_avg_spend_in          0
log_avg_balance_6m        0
log_avg_net_revenue_6m    0
log_avg_txn_cnt_out       0
in_amt_log                0
dtype: int64

сохранено features_v2.pkl
          share_pos    share_ecom     share_atm
count  19755.000000  19755.000000  19755.000000
mean       0.504371      0.228553      0.264798
std        0.178086      0.201321      0.166809
min        0.000000      0.000000      0.000000
25%        0.392467      0.063859  

In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

features2 = pd.read_pickle('features_v2.pkl')

behavior_cols_v2 = [
    'log_avg_spend_out', 'log_avg_txn_cnt_out', 'log_avg_balance_6m', 'log_avg_net_revenue_6m',
    'share_pos', 'share_ecom', 'share_atm', 'in_amt_log', 'n_products'
]

X = features2[behavior_cols_v2].copy()
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = np.clip(X_scaled, -5, 5)

X_scaled_df = pd.DataFrame(X_scaled, columns=behavior_cols_v2, index=features2['customer_id'])
X_scaled_df.to_pickle('X_v2_scaled.pkl')

print("Экстремальные значения (|z|>5) после клипа - должно быть 0:")
print((np.abs(X_scaled) >= 5).sum(axis=0))

results = []
for k in range(3, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels, sample_size=5000, random_state=42)
    db = davies_bouldin_score(X_scaled, labels)
    sizes = pd.Series(labels).value_counts().sort_index().tolist()
    results.append({'k': k, 'inertia': km.inertia_, 'silhouette': sil, 'davies_bouldin': db, 'cluster_sizes': sizes})

results_df = pd.DataFrame(results)
pd.set_option('display.width', 160)
print("\n" + results_df.to_string(index=False))
results_df.to_pickle('k_selection_v2.pkl')

Экстремальные значения (|z|>5) после клипа - должно быть 0:
[45  0  0  0  0  0  0  0 37]


/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: divide by zero encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: overflow encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/alishersabirov


 k      inertia  silhouette  davies_bouldin                                    cluster_sizes
 3 66291.389609    0.226928        1.470088                               [9419, 2777, 7559]
 4 57289.692640    0.219549        1.508608                         [2461, 4743, 4888, 7663]
 5 52537.136816    0.209299        1.542362                   [3897, 7004, 3267, 2397, 3190]
 6 48484.315436    0.194266        1.485235             [5471, 2764, 2527, 2326, 3072, 3595]
 7 44998.821631    0.201394        1.461496       [3517, 2526, 3098, 5408, 2760, 1388, 1058]
 8 42018.007435    0.209958        1.436849 [2509, 2665, 2302, 1068, 1388, 2324, 5471, 2028]


/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklear

In [9]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans

X_scaled_df = pd.read_pickle('X_v2_scaled.pkl')
features2 = pd.read_pickle('features_v2.pkl')
X = X_scaled_df.values

km = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = km.fit_predict(X)
features2['cluster_v2'] = labels

raw_cols = ['avg_spend_out', 'avg_spend_in', 'avg_balance_6m', 'avg_net_revenue_6m',
            'avg_txn_cnt_out', 'share_pos', 'share_ecom', 'share_atm', 'n_products']

profile = features2.groupby('cluster_v2')[raw_cols].median()
profile['n_clients'] = features2.groupby('cluster_v2').size()
profile['pct'] = (profile['n_clients'] / len(features2) * 100).round(1)

pd.set_option('display.width', 180)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print(profile.to_string())

features2.to_pickle('features_v2_clustered.pkl')

            avg_spend_out  avg_spend_in  avg_balance_6m  avg_net_revenue_6m  avg_txn_cnt_out  share_pos  share_ecom  share_atm  n_products  n_clients   pct
cluster_v2                                                                                                                                                 
0              259,628.40          0.00      322,717.00            3,225.17             1.75       0.61        0.05       0.23        1.00       2461 12.50
1            9,190,592.00  1,164,496.33    2,303,110.33           83,774.50            43.83       0.41        0.43       0.13        1.00       4743 24.00
2           11,123,430.58  2,716,953.17    6,878,014.17          146,126.83            28.00       0.51        0.25       0.21        3.00       4888 24.70
3            3,015,054.50    423,069.67    1,095,383.33           21,493.00            12.33       0.53        0.08       0.37        1.00       7663 38.80


/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: divide by zero encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: overflow encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/alishersabirov

In [10]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

X_scaled_df = pd.read_pickle('X_v2_scaled.pkl')
features2 = pd.read_pickle('features_v2_clustered.pkl')
X_base = X_scaled_df.values
base_labels = features2['cluster_v2'].values
behavior_cols_v2 = X_scaled_df.columns.tolist()

results = []
for rs in [0, 7, 99]:
    km = KMeans(n_clusters=4, random_state=rs, n_init=10)
    labels = km.fit_predict(X_base)
    results.append({'test': f'random_state={rs}', 'ARI': adjusted_rand_score(base_labels, labels)})

np.random.seed(1)
half_idx = np.random.choice(len(X_base), size=len(X_base)//2, replace=False)
km_half = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_half = km_half.fit_predict(X_base[half_idx])
results.append({'test': 'половина выборки', 'ARI': adjusted_rand_score(base_labels[half_idx], labels_half)})

# без share_atm и n_products (проверка чувствительности к отдельным признакам)
alt_cols = ['log_avg_spend_out', 'log_avg_txn_cnt_out', 'log_avg_balance_6m', 'log_avg_net_revenue_6m',
            'share_pos', 'share_ecom', 'in_amt_log']
from sklearn.preprocessing import RobustScaler
X_alt = np.clip(RobustScaler().fit_transform(features2[alt_cols]), -5, 5)
km_alt = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_alt = km_alt.fit_predict(X_alt)
results.append({'test': 'без share_atm и n_products', 'ARI': adjusted_rand_score(base_labels, labels_alt)})

# без in_amt_log (новый признак) - проверим его вклад отдельно
alt_cols2 = [c for c in behavior_cols_v2 if c != 'in_amt_log']
X_alt2 = np.clip(RobustScaler().fit_transform(features2[alt_cols2]), -5, 5)
km_alt2 = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_alt2 = km_alt2.fit_predict(X_alt2)
results.append({'test': 'без in_amt_log', 'ARI': adjusted_rand_score(base_labels, labels_alt2)})

stability_v2 = pd.DataFrame(results)
print(stability_v2.to_string(index=False))
stability_v2.to_pickle('stability_v2.pkl')

/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: divide by zero encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: overflow encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/alishersabirov

                      test  ARI
            random_state=0 0.99
            random_state=7 0.99
           random_state=99 0.99
          половина выборки 0.98
без share_atm и n_products 0.58
            без in_amt_log 0.97


In [11]:
import pandas as pd
import numpy as np
from sklearn.metrics import adjusted_rand_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

features2 = pd.read_pickle('features_v2_clustered.pkl')
X_scaled_df = pd.read_pickle('X_v2_scaled.pkl')

# --- квартили по обороту ---
features2['quartile_baseline'] = pd.qcut(features2['avg_spend_out'], q=4, labels=False, duplicates='drop')
ari_vs_quartile = adjusted_rand_score(features2['cluster_v2'], features2['quartile_baseline'])
print("ARI: типология v2 vs квартили по обороту:", ari_vs_quartile)

compare_cols = ['share_pos', 'share_ecom', 'share_atm', 'n_products', 'avg_balance_6m', 'avg_spend_in']
def within_group_variance_ratio(df, group_col, cols):
    total_var = df[cols].var()
    within_var = df.groupby(group_col)[cols].var().mean()
    return (within_var / total_var).mean()

ratio_ours = within_group_variance_ratio(features2, 'cluster_v2', compare_cols)
ratio_quartile = within_group_variance_ratio(features2, 'quartile_baseline', compare_cols)
print(f"Необъяснённая дисперсия - наша типология: {ratio_ours:.3f}, квартили: {ratio_quartile:.3f}")

# --- правило для нового клиента ---
X = X_scaled_df.values
y = features2['cluster_v2'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

knn = KNeighborsClassifier(n_neighbors=15)
knn.fit(X_train, y_train)
print("\nkNN accuracy:", accuracy_score(y_test, knn.predict(X_test)))

centroids = pd.DataFrame(X_train, columns=X_scaled_df.columns)
centroids['cluster'] = y_train
centroid_means = centroids.groupby('cluster').mean().values

def assign_to_nearest_centroid(x, centroids):
    return np.argmin(np.linalg.norm(centroids - x, axis=1))

y_pred_centroid = np.array([assign_to_nearest_centroid(x, centroid_means) for x in X_test])
print("Centroid-rule accuracy:", accuracy_score(y_test, y_pred_centroid))

features2.to_pickle('features_v2_final.pkl')

ARI: типология v2 vs квартили по обороту: 0.21348565717424411
Необъяснённая дисперсия - наша типология: 0.831, квартили: 0.918

kNN accuracy: 0.9744368514300177
Centroid-rule accuracy: 0.9972158947102


In [12]:
import pandas as pd

features2 = pd.read_pickle('features_v2_final.pkl')

names_v2 = {
    0: 'Спящие',
    1: 'Онлайн-активные',
    2: 'Премиум с высокими поступлениями',
    3: 'Наличностники',
}
features2['segment_name'] = features2['cluster_v2'].map(names_v2)

summary_v2 = features2.groupby('segment_name').agg(
    n_clients=('customer_id', 'count'),
    total_monthly_revenue=('avg_net_revenue_6m', 'sum'),
    avg_revenue_per_client=('avg_net_revenue_6m', 'mean'),
    median_balance=('avg_balance_6m', 'median'),
    median_spend=('avg_spend_out', 'median'),
    median_in=('avg_spend_in', 'median'),
    avg_products=('n_products', 'mean'),
).round(0)
summary_v2['pct_of_base'] = (summary_v2['n_clients'] / len(features2) * 100).round(1)
summary_v2['pct_of_revenue'] = (summary_v2['total_monthly_revenue'] / summary_v2['total_monthly_revenue'].sum() * 100).round(1)

pd.set_option('display.width', 180)
pd.set_option('display.float_format', lambda x: f'{x:,.0f}')
print(summary_v2.to_string())

features2.to_pickle('features_v2_named.pkl')
summary_v2.to_pickle('segment_summary_v2.pkl')

                                  n_clients  total_monthly_revenue  avg_revenue_per_client  median_balance  median_spend  median_in  avg_products  pct_of_base  pct_of_revenue
segment_name                                                                                                                                                                  
Наличностники                          7663            219,222,504                  28,608       1,095,383     3,015,054    423,070             1           39              11
Онлайн-активные                        4743            621,176,249                 130,967       2,303,110     9,190,592  1,164,496             1           24              31
Премиум с высокими поступлениями       4888          1,170,053,700                 239,373       6,878,014    11,123,431  2,716,953             3           25              58
Спящие                                 2461              9,952,981                   4,044         322,717       259,628     

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

features2 = pd.read_pickle('features_v2_named.pkl')
X_scaled_df = pd.read_pickle('X_v2_scaled.pkl')
stability_v2 = pd.read_pickle('stability_v2.pkl')
summary_v2 = pd.read_pickle('segment_summary_v2.pkl')

colors = {'Наличностники':'#4C72B0','Онлайн-активные':'#DD8452','Спящие':'#999999','Премиум с высокими поступлениями':'#55A868'}

# 1. PCA scatter
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled_df.values)
features2['pca1'] = X_pca[:,0]
features2['pca2'] = X_pca[:,1]

fig, ax = plt.subplots(figsize=(9,7))
for name, grp in features2.groupby('segment_name'):
    ax.scatter(grp['pca1'], grp['pca2'], s=6, alpha=0.4, label=name, color=colors.get(name))
ax.set_title(f'Кластеры клиентов в проекции PCA (объяснено {pca.explained_variance_ratio_.sum()*100:.0f}% дисперсии)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.legend(markerscale=4, fontsize=8)
plt.tight_layout()
plt.savefig('pca_scatter_v2.png', dpi=150)
plt.close()

# 2. boxplots
fig, axes = plt.subplots(1, 3, figsize=(16,5))
plot_cols = ['avg_spend_out', 'avg_spend_in', 'n_products']
titles = ['Траты (мес, сум)', 'Входящие переводы (мес, сум)', 'Кол-во продуктов']
order = ['Спящие','Наличностники','Онлайн-активные','Премиум с высокими поступлениями']
for ax, col, title in zip(axes, plot_cols, titles):
    data = [features2[features2['segment_name']==seg][col].clip(upper=features2[col].quantile(0.97)) for seg in order]
    ax.boxplot(data, labels=order, showfliers=False)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('segment_boxplots_v2.png', dpi=150)
plt.close()

# 3. stability chart
fig, ax = plt.subplots(figsize=(8,5))
bars = ax.bar(stability_v2['test'], stability_v2['ARI'], color='#4C72B0')
ax.axhline(0.7, color='red', linestyle='--', linewidth=1, label='порог приемлемой устойчивости (0.7)')
ax.set_ylim(0,1.05)
ax.set_ylabel('ARI')
ax.set_title('Устойчивость типологии v2 (после чистки p2p/cash-atm)')
plt.xticks(rotation=20, ha='right')
ax.legend()
for b in bars:
    val = b.get_height()
    ax.text(b.get_x()+b.get_width()/2, val+0.02, f'{val:.2f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('stability_chart_v2.png', dpi=150)
plt.close()

# 4. revenue share
fig, ax = plt.subplots(figsize=(8,5))
x = np.arange(len(summary_v2))
w = 0.35
ax.bar(x-w/2, summary_v2['pct_of_base'], w, label='% клиентской базы', color='#4C72B0')
ax.bar(x+w/2, summary_v2['pct_of_revenue'], w, label='% доходности банка', color='#DD8452')
ax.set_xticks(x); ax.set_xticklabels(summary_v2.index, rotation=20, ha='right')
ax.set_ylabel('%')
ax.set_title('Доля базы vs доля дохода по сегментам (v2)')
ax.legend()
plt.tight_layout()
plt.savefig('revenue_share_chart_v2.png', dpi=150)
plt.close()

print("Готово: 4 графика сохранены")

/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: overflow encountered in matmul
  X_transformed = X @ self.components_.T
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: invalid value encountered in matmul
  X_transformed = X @ self.components_.T
/var/folders/ch/fyp9js953h77f_wvqmdx9n280000gn/T/ipykernel_7987/1935901544.py:36: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=order, showfliers=False)
/var/folders/ch/fyp9js953h77f_wvqmdx9n280000gn/T/ipyke

Готово: 4 графика сохранены


In [14]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

X_scaled_df = pd.read_pickle('X_v2_scaled.pkl')
features2 = pd.read_pickle('features_v2_named.pkl')
X = X_scaled_df.values

km4 = KMeans(n_clusters=4, random_state=42, n_init=10)
labels4 = km4.fit_predict(X)

km5 = KMeans(n_clusters=5, random_state=42, n_init=10)
labels5 = km5.fit_predict(X)

features2['k4'] = labels4
features2['k5'] = labels5

raw_cols = ['avg_spend_out', 'avg_spend_in', 'avg_balance_6m', 'avg_net_revenue_6m',
            'avg_txn_cnt_out', 'share_pos', 'share_ecom', 'share_atm', 'n_products']

print("=== Профиль k=5 ===")
profile5 = features2.groupby('k5')[raw_cols].median()
profile5['n'] = features2.groupby('k5').size()
profile5['pct'] = (profile5['n'] / len(features2) * 100).round(1)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print(profile5.to_string())

# как k=5 соотносится с k=4 - какой кластер k=4 раздробился?
print("\n=== Таблица сопряжённости k=4 x k=5 ===")
print(pd.crosstab(features2['k4'], features2['k5']))

print("\nARI между k=4 и k=5:", adjusted_rand_score(labels4, labels5))

/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: divide by zero encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: overflow encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/alishersabirov

=== Профиль k=5 ===
    avg_spend_out  avg_spend_in  avg_balance_6m  avg_net_revenue_6m  avg_txn_cnt_out  share_pos  share_ecom  share_atm  n_products     n   pct
k5                                                                                                                                            
0   10,835,611.67  3,073,172.33    5,908,280.83          124,655.17            24.67       0.57        0.21       0.20        1.00  3897 19.70
1    2,812,105.50    342,039.75      999,364.25           19,649.75            12.17       0.52        0.08       0.38        1.00  7004 35.50
2   10,853,317.67  2,641,954.50    5,757,909.83          138,221.67            31.00       0.49        0.27       0.21        3.00  3267 16.50
3      253,529.00          0.00      319,165.83            3,170.67             1.67       0.61        0.05       0.22        1.00  2397 12.10
4    7,810,831.00    806,244.25    1,776,765.00           68,480.42            45.58       0.33        0.53       0.11    

In [25]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import RobustScaler

X_scaled_df = pd.read_pickle('X_v2_scaled.pkl')
features2 = pd.read_pickle('features_v2_named.pkl')
ages = pd.read_pickle('ages.pkl')
X = X_scaled_df.values
behavior_cols_v2 = X_scaled_df.columns.tolist()

km5 = KMeans(n_clusters=5, random_state=42, n_init=10)
labels5 = km5.fit_predict(X)
features2['cluster_v3'] = labels5
base_labels = labels5

# --- имена сегментов k=5 (по профилю из прошлого шага) ---
names_v3 = {
    0: 'Высокодоходные без допродажи',
    1: 'Наличностники',
    2: 'Премиум мульти-продуктовые',
    3: 'Спящие',
    4: 'Онлайн-активные',
}
features2['segment_name_v3'] = features2['cluster_v3'].map(names_v3)

# --- устойчивость ---
stability_results = []
for rs in [0, 7, 99]:
    km = KMeans(n_clusters=5, random_state=rs, n_init=10)
    labels = km.fit_predict(X)
    stability_results.append({'test': f'random_state={rs}', 'ARI': adjusted_rand_score(base_labels, labels)})

np.random.seed(1)
half_idx = np.random.choice(len(X), size=len(X)//2, replace=False)
km_half = KMeans(n_clusters=5, random_state=42, n_init=10)
labels_half = km_half.fit_predict(X[half_idx])
stability_results.append({'test': 'половина выборки', 'ARI': adjusted_rand_score(base_labels[half_idx], labels_half)})

alt_cols = [c for c in behavior_cols_v2 if c not in ('share_atm', 'n_products')]
X_alt = np.clip(RobustScaler().fit_transform(features2[alt_cols]), -5, 5)
km_alt = KMeans(n_clusters=5, random_state=42, n_init=10)
labels_alt = km_alt.fit_predict(X_alt)
stability_results.append({'test': 'без share_atm и n_products', 'ARI': adjusted_rand_score(base_labels, labels_alt)})

alt_cols2 = [c for c in behavior_cols_v2 if c != 'in_amt_log']
X_alt2 = np.clip(RobustScaler().fit_transform(features2[alt_cols2]), -5, 5)
km_alt2 = KMeans(n_clusters=5, random_state=42, n_init=10)
labels_alt2 = km_alt2.fit_predict(X_alt2)
stability_results.append({'test': 'без in_amt_log', 'ARI': adjusted_rand_score(base_labels, labels_alt2)})

stability_v3 = pd.DataFrame(stability_results)
print("=== Устойчивость k=5 ===")
print(stability_v3.to_string(index=False))

# --- квартили ---
features2['quartile_baseline'] = pd.qcut(features2['avg_spend_out'], q=4, labels=False, duplicates='drop')
ari_q = adjusted_rand_score(features2['cluster_v3'], features2['quartile_baseline'])
compare_cols = ['share_pos', 'share_ecom', 'share_atm', 'n_products', 'avg_balance_6m', 'avg_spend_in']
def wgv(df, gc, cols):
    return (df.groupby(gc)[cols].var().mean() / df[cols].var()).mean()
print(f"\nARI vs квартили: {ari_q:.3f}")
print(f"Необъясн. дисперсия - типология: {wgv(features2,'cluster_v3',compare_cols):.3f}, квартили: {wgv(features2,'quartile_baseline',compare_cols):.3f}")

# --- правило для нового клиента ---
y = features2['cluster_v3'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
knn = KNeighborsClassifier(n_neighbors=15).fit(X_train, y_train)
print("\nkNN accuracy:", accuracy_score(y_test, knn.predict(X_test)))
centroids = pd.DataFrame(X_train, columns=behavior_cols_v2); centroids['cluster'] = y_train
cm = centroids.groupby('cluster').mean().values
pred_centroid = np.array([np.argmin(np.linalg.norm(cm - x, axis=1)) for x in X_test])
print("Centroid accuracy:", accuracy_score(y_test, pred_centroid))

# --- деньги ---
summary_v3 = features2.groupby('segment_name_v3').agg(
    n_clients=('customer_id','count'), total_rev=('avg_net_revenue_6m','sum'),
    median_balance=('avg_balance_6m','median'), median_spend=('avg_spend_out','median'),
    avg_products=('n_products','mean')
).round(0)
summary_v3['pct_of_base'] = (summary_v3['n_clients']/len(features2)*100).round(1)
summary_v3['pct_of_revenue'] = (summary_v3['total_rev']/summary_v3['total_rev'].sum()*100).round(1)
print("\n=== Деньги по сегментам k=5 ===")
pd.set_option('display.width', 180)
print(summary_v3.to_string())

# --- возраст по сегментам ---
features_age = features2.merge(ages[['customer_id','age']], on='customer_id', how='left')
print("\n=== Возраст по сегментам ===")
print(features_age.groupby('segment_name_v3')['age'].describe()[['count','mean','50%','min','max']])

features2.to_pickle('features_v3_named.pkl')
features_age.to_pickle('features_v3_age.pkl')
stability_v3.to_pickle('stability_v3.pkl')
summary_v3.to_pickle('segment_summary_v3.pkl')
X_scaled_df.to_pickle('X_v3_scaled.pkl')  # то же самое, просто для симметрии именования

/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: divide by zero encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: overflow encountered in matmul
  current_pot = closest_dist_sq @ sample_weight
/Users/alishersabirov

=== Устойчивость k=5 ===
                      test  ARI
            random_state=0 1.00
            random_state=7 1.00
           random_state=99 1.00
          половина выборки 0.98
без share_atm и n_products 0.61
            без in_amt_log 0.97

ARI vs квартили: 0.210
Необъясн. дисперсия - типология: 0.791, квартили: 0.918

kNN accuracy: 0.9716527461402177
Centroid accuracy: 0.9987344975955454

=== Деньги по сегментам k=5 ===
                              n_clients      total_rev  median_balance  median_spend  avg_products  pct_of_base  pct_of_revenue
segment_name_v3                                                                                                                
Высокодоходные без допродажи       3897 808,142,517.00    5,908,281.00 10,835,612.00          1.00        19.70           40.00
Наличностники                      7004 160,102,538.00      999,364.00  2,812,106.00          1.00        35.50            7.90
Онлайн-активные                    3190 312,145,703.00

In [26]:
import pandas as pd
import numpy as np

features_age = pd.read_pickle('features_v3_age.pkl')

# корреляция возраста с ключевыми числовыми признаками
corr_cols = ['age', 'avg_spend_out', 'avg_spend_in', 'avg_balance_6m', 'n_products', 'share_ecom', 'share_atm']
print("Корреляция возраста с признаками:")
print(features_age[corr_cols].corr()['age'].sort_values())

# возрастные бакеты x сегменты (% внутри бакета)
features_age['age_bracket'] = pd.cut(features_age['age'], bins=[18,25,35,45,55,60], labels=['19-25','26-35','36-45','46-55','56+'])
crosstab = pd.crosstab(features_age['age_bracket'], features_age['segment_name_v3'], normalize='index') * 100
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f'{x:.1f}')
print("\nРаспределение сегментов по возрастным группам (% внутри группы):")
print(crosstab.to_string())

# для графика: гистограмма возраста по сегментам
features_age[['segment_name_v3','age']].to_pickle('age_for_plot.pkl')

Корреляция возраста с признаками:
avg_spend_in     -0.00
n_products       -0.00
avg_balance_6m   -0.00
share_atm        -0.00
avg_spend_out     0.00
share_ecom        0.00
age               1.00
Name: age, dtype: float64

Распределение сегментов по возрастным группам (% внутри группы):
segment_name_v3  Высокодоходные без допродажи  Наличностники  Онлайн-активные  Премиум мульти-продуктовые  Спящие
age_bracket                                                                                                      
19-25                                    20.2           34.8             15.2                        17.2    12.5
26-35                                    20.2           35.5             15.9                        16.1    12.3
36-45                                    19.4           34.9             16.8                        17.3    11.6
46-55                                    18.9           36.5             16.4                        15.9    12.4
56+                          

In [27]:
import pandas as pd

features_age = pd.read_pickle('features_v3_age.pkl')
features_age['age_bracket'] = pd.cut(features_age['age'], bins=[18,25,35,45,55,60], labels=['19-25','26-35','36-45','46-55','56+'])

crosstab = pd.crosstab(features_age['age_bracket'], features_age['segment_name_v3'], normalize='index') * 100
pd.set_option('display.float_format', lambda x: f'{x:.1f}')

# транспонируем и печатаем построчно, чтобы не обрезалось
print(crosstab.T.to_string())

print("\nРазмер каждой возрастной группы:")
print(features_age['age_bracket'].value_counts().sort_index())

age_bracket                   19-25  26-35  36-45  46-55  56+
segment_name_v3                                              
Высокодоходные без допродажи   20.2   20.2   19.4   18.9 21.9
Наличностники                  34.8   35.5   34.9   36.5 34.8
Онлайн-активные                15.2   15.9   16.8   16.4 15.9
Премиум мульти-продуктовые     17.2   16.1   17.3   15.9 16.5
Спящие                         12.5   12.3   11.6   12.4 11.0

Размер каждой возрастной группы:
age_bracket
19-25    3225
26-35    5192
36-45    5116
46-55    5220
56+      1002
Name: count, dtype: int64


In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

features3 = pd.read_pickle('features_v3_named.pkl')
X_scaled_df = pd.read_pickle('X_v3_scaled.pkl')
stability_v3 = pd.read_pickle('stability_v3.pkl')
summary_v3 = pd.read_pickle('segment_summary_v3.pkl')
features_age = pd.read_pickle('features_v3_age.pkl')

colors = {
    'Наличностники':'#4C72B0','Онлайн-активные':'#DD8452',
    'Премиум мульти-продуктовые':'#55A868','Спящие':'#999999',
    'Высокодоходные без допродажи':'#C44E52'
}
order = ['Спящие','Наличностники','Онлайн-активные','Высокодоходные без допродажи','Премиум мульти-продуктовые']

# 1. PCA scatter (5 clusters)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled_df.values)
features3['pca1'] = X_pca[:,0]
features3['pca2'] = X_pca[:,1]

fig, ax = plt.subplots(figsize=(9,7))
for name, grp in features3.groupby('segment_name_v3'):
    ax.scatter(grp['pca1'], grp['pca2'], s=6, alpha=0.4, label=name, color=colors.get(name))
ax.set_title(f'Кластеры клиентов в проекции PCA (объяснено {pca.explained_variance_ratio_.sum()*100:.0f}% дисперсии)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.legend(markerscale=4, fontsize=8)
plt.tight_layout()
plt.savefig('pca_scatter_v3.png', dpi=150)
plt.close()

# 2. boxplots
fig, axes = plt.subplots(1, 3, figsize=(16,5))
plot_cols = ['avg_spend_out', 'avg_balance_6m', 'n_products']
titles = ['Траты (мес, сум)', 'Средний баланс (сум)', 'Кол-во продуктов']
for ax, col, title in zip(axes, plot_cols, titles):
    data = [features3[features3['segment_name_v3']==seg][col].clip(upper=features3[col].quantile(0.97)) for seg in order]
    ax.boxplot(data, labels=order, showfliers=False)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('segment_boxplots_v3.png', dpi=150)
plt.close()

# 3. stability chart
fig, ax = plt.subplots(figsize=(8,5))
bars = ax.bar(stability_v3['test'], stability_v3['ARI'], color='#4C72B0')
ax.axhline(0.7, color='red', linestyle='--', linewidth=1, label='порог приемлемой устойчивости (0.7)')
ax.set_ylim(0,1.05)
ax.set_ylabel('ARI')
ax.set_title('Устойчивость типологии (k=5)')
plt.xticks(rotation=20, ha='right')
ax.legend()
for b in bars:
    val = b.get_height()
    ax.text(b.get_x()+b.get_width()/2, val+0.02, f'{val:.2f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('stability_chart_v3.png', dpi=150)
plt.close()

# 4. revenue share
fig, ax = plt.subplots(figsize=(9,5.5))
summary_v3 = summary_v3.reindex(order)
x = np.arange(len(summary_v3))
w = 0.35
ax.bar(x-w/2, summary_v3['pct_of_base'], w, label='% клиентской базы', color='#4C72B0')
ax.bar(x+w/2, summary_v3['pct_of_revenue'], w, label='% доходности банка', color='#DD8452')
ax.set_xticks(x); ax.set_xticklabels(summary_v3.index, rotation=20, ha='right')
ax.set_ylabel('%')
ax.set_title('Доля базы vs доля дохода по сегментам (k=5)')
ax.legend()
plt.tight_layout()
plt.savefig('revenue_share_chart_v3.png', dpi=150)
plt.close()

# 5. age distribution by segment
fig, ax = plt.subplots(figsize=(10,5.5))
data = [features_age[features_age['segment_name_v3']==seg]['age'].dropna() for seg in order]
bp = ax.boxplot(data, labels=order, showfliers=False, patch_artist=True)
for patch, seg in zip(bp['boxes'], order):
    patch.set_facecolor(colors.get(seg))
    patch.set_alpha(0.5)
ax.set_title('Возраст клиента практически не отличается между сегментами')
ax.set_ylabel('Возраст, лет')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig('age_by_segment.png', dpi=150)
plt.close()

print("Готово: 5 графиков сохранены")

/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: overflow encountered in matmul
  X_transformed = X @ self.components_.T
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: invalid value encountered in matmul
  X_transformed = X @ self.components_.T
/var/folders/ch/fyp9js953h77f_wvqmdx9n280000gn/T/ipykernel_7987/4121544810.py:41: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=order, showfliers=False)
/var/folders/ch/fyp9js953h77f_wvqmdx9n280000gn/T/ipyke

Готово: 5 графиков сохранены


/var/folders/ch/fyp9js953h77f_wvqmdx9n280000gn/T/ipykernel_7987/4121544810.py:82: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data, labels=order, showfliers=False, patch_artist=True)


In [29]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

features3 = pd.read_pickle('features_v3_named.pkl')
X_scaled_df = pd.read_pickle('X_v3_scaled.pkl')

colors = {
    'Наличностники':'#4C72B0','Онлайн-активные':'#DD8452',
    'Премиум мульти-продуктовые':'#55A868','Спящие':'#999999',
    'Высокодоходные без допродажи':'#C44E52'
}

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled_df.values)
features3['pca1'] = X_pca[:,0]
features3['pca2'] = X_pca[:,1]

fig, ax = plt.subplots(figsize=(9,7))
for name, grp in features3.groupby('segment_name_v3'):
    ax.scatter(grp['pca1'], grp['pca2'], s=6, alpha=0.4, label=name, color=colors.get(name))
ax.set_title(f'Кластеры клиентов в проекции PCA (объяснено {pca.explained_variance_ratio_.sum()*100:.0f}% дисперсии)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.legend(markerscale=4, fontsize=8)
plt.tight_layout()
plt.savefig('pca_scatter_v3.png', dpi=150)
plt.close()
print("сохранено pca_scatter_v3.png")

сохранено pca_scatter_v3.png


/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: overflow encountered in matmul
  X_transformed = X @ self.components_.T
/Users/alishersabirov252gmail.com/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: invalid value encountered in matmul
  X_transformed = X @ self.components_.T


In [24]:
import pandas as pd
from sqlalchemy import text

with engine.connect() as conn:
    ages = pd.read_sql(text("""
        SELECT customer_id, birth_date,
               DATE_PART('year', AGE(DATE '2025-12-28', birth_date)) AS age
        FROM seg_customers
    """), conn)

print(ages['age'].describe())
print("\nПропуски в birth_date:", ages['birth_date'].isna().sum())
ages.to_pickle('ages.pkl')
print("сохранено ages.pkl")

count   20,000.00
mean        38.33
std         11.02
min         19.00
25%         29.00
50%         38.00
75%         48.00
max         57.00
Name: age, dtype: float64

Пропуски в birth_date: 0
сохранено ages.pkl
